In [1]:
# single cell libraries
import scanpy as sc
import anndata as ad
import scrublet as scr
import leidenalg, igraph
import bbknn

# processing tools
import numpy as np
import pandas as pd
import scipy

# plotting tools
import matplotlib.pyplot as pl

# file grab
from pathlib import Path
import pooch

/root/miniconda/envs/unified_env/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
/root/miniconda/envs/unified_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sc.settings.verbosity = 2
sc.logging.print_versions()
results_file = "./write/_"
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(3, 3), facecolor="white")

In [3]:
adata = sc.datasets.paul15()
adata

AnnData object with n_obs × n_vars = 2730 × 3451
    obs: 'paul15_clusters'
    uns: 'iroot'

In [4]:
adata.X = adata.X.astype("float32")

In [5]:
#QC

# mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("mt-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True
)

sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

In [6]:
print(adata.var_names[adata.var["mt"]].tolist())
print(adata.var["mt"].sum())

# mitochondrial genes seem to be filtered out in advance

[]
0


In [7]:
# Filter minimum counts
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=5) 

# Check zero counts
total_counts = adata.X.sum(axis=1).A1 if scipy.sparse.issparse(adata.X) else adata.X.sum(axis=1)
if np.any(total_counts <= 1):
    print("Warning: Cells with zero counts detected. Filtering...")
    adata = adata[total_counts > 1, :]

# Check NaN
X = adata.X.toarray() if scipy.sparse.issparse(adata.X) else adata.X
if np.any(np.isnan(X)) or np.any(np.isinf(X)):
    print("NaNs/Infs post-normalization. Check data.")
    raise

# Doublet count
scrub = scr.Scrublet(adata.X, expected_doublet_rate=0.1)  # Adjust for epilepsy
try:
    doublet_scores, predicted_doublets = scrub.scrub_doublets()
except ValueError as e:
    print(f"Scrublet error: {e}. Checking adata.X...")
    if np.any(np.isnan(X)) or np.any(np.isinf(X)):
        print("NaNs/Infs in adata.X.")
    raise
adata.obs['doublet_score'] = doublet_scores
adata.obs['predicted_doublet'] = predicted_doublets
adata = adata[~adata.obs['predicted_doublet'], :]

# Re-check zero counts post-Scrublet
total_counts = adata.X.sum(axis=1).A1 if scipy.sparse.issparse(adata.X) else adata.X.sum(axis=1)
if np.any(total_counts <= 1):
    print(f"Warning: {np.sum(total_counts <= 1)} cells with <=1 counts post-Scrublet. Filtering...")
    adata = adata[total_counts > 1, :]


filtered out 2 genes that are detected in less than 5 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.56
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 26.2%
Overall doublet rate:
	Expected   = 10.0%
	Estimated  = 0.6%
Elapsed time: 12.3 seconds


In [8]:
# Save count data
adata.layers[count] = adata.X.copy()

# First normalization to handle dispersion-based HVG
sc.pp.normalize_total(adata)

# HVG selection
sc.pp.highly_variable_genes(adata, n_top_genes=1000, flavor='seurat_v3', min_mean=0.01, max_mean=10)

# Subset to HVGs
adata = adata[:, adata.var.highly_variable]

# Set raw
X = adata.X.toarray() if scipy.sparse.issparse(adata.X) else adata.X
if np.any(np.isnan(X)) or np.any(np.isinf(X)):
    print("NaNs/Infs post-normalization. Check data.")
    raise
adata.raw = adata

normalizing counts per cell
    finished (0:00:00)
extracting highly variable genes
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)


/root/miniconda/envs/unified_env/lib/python3.10/site-packages/scanpy/preprocessing/_normalization.py:269: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)
/root/miniconda/envs/unified_env/lib/python3.10/site-packages/legacy_api_wrap/__init__.py:82: UserWarning: If you pass `n_top_genes`, all cutoffs are ignored.
  return fn(*args_all, **kw)
/root/miniconda/envs/unified_env/lib/python3.10/site-packages/scanpy/preprocessing/_normalization.py:269: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


In [9]:
# downstream
sc.pp.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, save="_1_prac_tutorial")

computing PCA
    with n_comps=50
    finished (0:00:21)
computing neighbors
    using 'X_pca' with n_pcs = 50
    finished (0:00:02)
computing UMAP
    finished (0:00:05)


In [10]:
# clustering
sc.tl.leiden(adata, flavor="igraph", n_iterations=2)
sc.pl.umap(adata, color=["leiden"], save="_2_prac_leiden")

running Leiden clustering
    finished (0:00:00)


In [11]:
# annotation
sc.tl.rank_genes_groups(adata, groupby='leiden', use_raw=True, method='wilcoxon')

ranking genes
    finished (0:00:00)


/root/miniconda/envs/unified_env/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/root/miniconda/envs/unified_env/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/root/miniconda/envs/unified_env/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/root/miniconda/envs/unified_env/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name, "logfoldchanges"] = np.log2(
/root/miniconda/envs/unified_env/lib/python3.10/site-packages/scanpy/tools/_rank_genes_groups.py:482: RuntimeWarning: invalid value encountered in log2
  self.stats[group_name,

In [12]:
# canonical marker genes
marker_genes = {
    "CD14+ Mono": ["FCN1", "CD14"],
    "CD16+ Mono": ["TCF7L2", "FCGR3A", "LYN"],
    # Note: DMXL2 should be negative
    "cDC2": ["CST3", "COTL1", "LYZ", "DMXL2", "CLEC10A", "FCER1A"],
    "Erythroblast": ["MKI67", "HBA1", "HBB"],
    # Note HBM and GYPA are negative markers
    "Proerythroblast": ["CDK6", "SYNGR1", "HBM", "GYPA"],
    "NK": ["GNLY", "NKG7", "CD247", "FCER1G", "TYROBP", "KLRG1", "FCGR3A"],
    "ILC": ["ID2", "PLCG2", "GNLY", "SYNE1"],
    "Naive CD20+ B": ["MS4A1", "IL4R", "IGHD", "FCRL1", "IGHM"],
    # Note IGHD and IGHM are negative markers
    "B cells": [
        "MS4A1",
        "ITGB1",
        "COL4A4",
        "PRDM1",
        "IRF4",
        "PAX5",
        "BCL11A",
        "BLK",
        "IGHD",
        "IGHM",
    ],
    "Plasma cells": ["MZB1", "HSP90B1", "FNDC3B", "PRDM1", "IGKC", "JCHAIN"],
    # Note PAX5 is a negative marker
    "Plasmablast": ["XBP1", "PRDM1", "PAX5"],
    "CD4+ T": ["CD4", "IL7R", "TRBC2"],
    "CD8+ T": ["CD8A", "CD8B", "GZMK", "GZMA", "CCL5", "GZMB", "GZMH", "GZMA"],
    "T naive": ["LEF1", "CCR7", "TCF7"],
    "pDC": ["GZMB", "IL3RA", "COBLL1", "TCF4"],
}

In [13]:
sc.pl.dotplot(adata, marker_genes, groupby='leiden', use_raw=True, dendrogram=True)

KeyError: "Could not find keys [np.str_('BCL11A'), np.str_('BLK'), np.str_('CCL5'), np.str_('CCR7'), np.str_('CD14'), np.str_('CD247'), np.str_('CD4'), np.str_('CD8A'), np.str_('CD8B'), np.str_('CDK6'), np.str_('CLEC10A'), np.str_('COBLL1'), np.str_('COL4A4'), np.str_('COTL1'), np.str_('CST3'), np.str_('DMXL2'), np.str_('FCER1A'), np.str_('FCER1G'), np.str_('FCGR3A'), np.str_('FCN1'), np.str_('FCRL1'), np.str_('FNDC3B'), np.str_('GNLY'), np.str_('GYPA'), np.str_('GZMA'), np.str_('GZMB'), np.str_('GZMH'), np.str_('GZMK'), np.str_('HBA1'), np.str_('HBB'), np.str_('HBM'), np.str_('HSP90B1'), np.str_('ID2'), np.str_('IGHD'), np.str_('IGHM'), np.str_('IGKC'), np.str_('IL3RA'), np.str_('IL4R'), np.str_('IL7R'), np.str_('IRF4'), np.str_('ITGB1'), np.str_('JCHAIN'), np.str_('KLRG1'), np.str_('LEF1'), np.str_('LYN'), np.str_('LYZ'), np.str_('MKI67'), np.str_('MS4A1'), np.str_('MZB1'), np.str_('NKG7'), np.str_('PAX5'), np.str_('PLCG2'), np.str_('PRDM1'), np.str_('SYNE1'), np.str_('SYNGR1'), np.s

In [ ]:
adata.write("cache/adata_practice_trajectory.h5ad")

In [ ]:
adata= sc.read_h5ad("cache/adata_practice_trajectory.h5ad")

In [ ]:
adata